In [54]:
from pathlib import Path
import pandas as pd
from datetime import datetime

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)

BASE_DIR = Path(r"C:\Users\user\Desktop\RAW_LOG")
MIDDLE_FOLDERS = ["TC6", "TC7", "TC8", "TC9", "Vision3"]
TARGET_FOLDERS = ["GoodFile", "BadFile"]

FCT_MAP = {
    "TC6": "FCT1",
    "TC7": "FCT2",
    "TC8": "FCT3",
    "TC9": "FCT4",
}

def classify_vision_equipment(file_path: Path):
    eq = "Vision?"
    tp = None
    try:
        with open(file_path, "r", encoding="cp949", errors="ignore") as f:
            line6 = None
            for i in range(6):
                line = f.readline()
                if not line:
                    break
                if i == 5:
                    line6 = line
                    break
        if line6 is not None:
            if "Test Program" in line6 and "LED1" in line6:
                eq, tp = "Vision1", "LED1"
            elif "Test Program" in line6 and "LED2" in line6:
                eq, tp = "Vision2", "LED2"
            else:
                eq = "Vision3"
    except Exception as e:
        eq, tp = "Vision3", f"ERROR: {e}"
    return eq, tp

records = []
count_files = 0

for mid in MIDDLE_FOLDERS:
    mid_path = BASE_DIR / mid
    if not mid_path.exists():
        print(f"[SKIP] {mid_path} 존재하지 않음")
        continue

    for date_folder in sorted(mid_path.iterdir()):
        if not date_folder.is_dir():
            continue

        for gb in TARGET_FOLDERS:
            gb_path = date_folder / gb
            if not gb_path.exists():
                continue

            for f in gb_path.iterdir():
                if not f.is_file():
                    continue

                count_files += 1
                name = f.name
                stem = f.stem
                ext = f.suffix
                length = len(stem)
                char18 = stem[17] if length >= 18 else ""

                if mid in FCT_MAP:
                    eq, tp = FCT_MAP[mid], None
                elif mid == "Vision3":
                    eq, tp = classify_vision_equipment(f)
                else:
                    eq, tp = mid, None

                records.append({
                    "equipment": eq,
                    "middle_folder": mid,
                    "date_folder": date_folder.name,
                    "good_bad": gb,
                    "filename": name,
                    "stem": stem,
                    "ext": ext,
                    "length": length,
                    "char18": char18,
                    "full_path": str(f),
                    "test_program": tp,
                })

df = pd.DataFrame(records)

print("\n===================== [a] RESULT =====================")
print(f"총 스캔 파일 개수 : {count_files}")


===================== [a] RESULT =====================
총 스캔 파일 개수 : 29282


In [55]:
if not df.empty:
    equip_counts = (
        df.groupby("equipment")["filename"]
          .count()
          .reset_index(name="file_count")
          .sort_values("equipment")
          .reset_index(drop=True)
    )
else:
    equip_counts = pd.DataFrame(columns=["equipment", "file_count"])

print("\n===================== [b] RESULT =====================")
print("[설비별 파일 개수]")
print(equip_counts)


===================== [b] RESULT =====================
[설비별 파일 개수]
  equipment  file_count
0      FCT1        3540
1      FCT2        3559
2      FCT3        3953
3      FCT4        3935
4   Vision1        7107
5   Vision2        7188


In [56]:
def make_path_label(row_dict):
    return f"{row_dict['equipment']}\\{row_dict['date_folder']}\\{row_dict['good_bad']}"

bad_length_records = []

for row in df.itertuples(index=False):
    stem = row.stem
    length = row.length
    char18 = row.char18

    reason = None

    if length < 18:
        reason = f"파일명 길이({length})<18 → 18번째 자리 없음"
    else:
        if char18 in ("C", "1"):
            if length != 51:
                reason = f"18번째 자리={char18}, 길이=51이어야 하나 실제={length}"
        elif char18 == "J":
            if length not in (51, 53):
                reason = f"18번째 자리=J, 길이=51 또는 53이어야 하나 실제={length}"
            else:
                if length == 53:
                    if len(stem) < 47 or stem[46] != "R":
                        reason = f"18번째 자리=J, 길이=53인데 47번째 자리!='R' (실제='{stem[46] if len(stem)>=47 else ''}')"
        elif char18 in ("P", "N"):
            if length != 52:
                reason = f"18번째 자리={char18}, 길이=52이어야 하나 실제={length}"
        elif char18 == "S":
            if length not in (52, 54):
                reason = f"18번째 자리=S, 길이=52 또는 54이어야 하나 실제={length}"
            else:
                if length == 54:
                    if len(stem) < 48 or stem[47] != "R":
                        reason = f"18번째 자리=S, 길이=54인데 48번째 자리!='R' (실제='{stem[47] if len(stem)>=48 else ''}')"
        else:
            reason = f"18번째 자리='{char18}' → 정의된 규칙 없음"

    if reason is not None:
        path_label = make_path_label(row._asdict())
        bad_length_records.append({
            "경로": path_label,
            "파일명": row.filename,
            "사유": f"[길이검증] {reason}",
        })

df_bad_length = pd.DataFrame(bad_length_records)

print("\n===================== [c] RESULT =====================")
print(f"파일명 길이 규칙 위반 건수: {len(df_bad_length)}")
print(df_bad_length.head(10))


===================== [c] RESULT =====================
파일명 길이 규칙 위반 건수: 0
Empty DataFrame
Columns: []
Index: []


In [57]:
# d) 파일명 날짜 vs 폴더명 날짜 비교
#    → 파일날짜 == 폴더날짜  또는 파일날짜 == 폴더날짜+1 인 경우만 정상으로 간주

date_mismatch_records = []

for row in df.itertuples(index=False):
    stem = row.stem
    length = row.length
    folder_date_str = row.date_folder

    # 1) 길이에 따른 날짜 위치
    if length == 51 or length == 53:
        # 32~39 → idx 31~38
        if len(stem) >= 39:
            name_date_str = stem[31:39]
        else:
            name_date_str = ""
    elif length == 52 or length == 54:
        # 33~40 → idx 32~39
        if len(stem) >= 40:
            name_date_str = stem[32:40]
        else:
            name_date_str = ""
    else:
        # 51~54가 아니면 날짜 추출 불가 (이미 c에서 잡히는 케이스)
        name_date_str = ""
        reason = f"[날짜검증] 파일명 길이({length})가 51~54 규칙 밖이라 날짜 추출 불가"
        path_label = make_path_label(row._asdict())
        date_mismatch_records.append({
            "경로": path_label,
            "파일명": row.filename,
            "사유": reason,
        })
        continue

    # 2) 날짜 문자열 형식 체크
    if len(name_date_str) != 8 or not name_date_str.isdigit() or not folder_date_str.isdigit():
        reason = f"[날짜검증] 파일명날짜('{name_date_str}') 또는 폴더날짜('{folder_date_str}') 형식 오류"
        path_label = make_path_label(row._asdict())
        date_mismatch_records.append({
            "경로": path_label,
            "파일명": row.filename,
            "사유": reason,
        })
        continue

    # 3) 날짜 파싱
    try:
        file_date = datetime.strptime(name_date_str, "%Y%m%d").date()
        folder_date = datetime.strptime(folder_date_str, "%Y%m%d").date()
    except Exception as e:
        reason = f"[날짜검증] 날짜 파싱 실패: file='{name_date_str}', folder='{folder_date_str}', error={e}"
        path_label = make_path_label(row._asdict())
        date_mismatch_records.append({
            "경로": path_label,
            "파일명": row.filename,
            "사유": reason,
        })
        continue

    # 4) 파일날짜 - 폴더날짜 차이 계산
    #    → 0일(같음) 또는 1일(파일날짜가 폴더보다 1일 크다)만 허용
    day_diff = (file_date - folder_date).days

    if day_diff in (0, 1):
        # 정상(폴더날짜 == 파일날짜 or 파일날짜 == 폴더날짜+1) → 패스
        continue
    else:
        # 그 외는 모두 불일치로 처리
        reason = f"[날짜검증] 폴더날짜({folder_date_str}) vs 파일날짜({name_date_str}) 차이 {day_diff}일"
        path_label = make_path_label(row._asdict())
        date_mismatch_records.append({
            "경로": path_label,
            "파일명": row.filename,
            "사유": reason,
        })

df_date_mismatch = pd.DataFrame(date_mismatch_records)

print(f"\n[d] 날짜 불일치(or 오류) 건수: {len(df_date_mismatch)}")
if not df_date_mismatch.empty:
    display(df_date_mismatch.head(20))


[d] 날짜 불일치(or 오류) 건수: 0


In [58]:
OUTPUT_EXCEL = BASE_DIR.parent / "a1_FCT_Vision_precheck_result.xlsx"

if df_bad_length.empty and df_date_mismatch.empty:
    sheet2_df = pd.DataFrame(columns=["경로", "파일명", "사유"])
else:
    sheet2_df = pd.concat([df_bad_length, df_date_mismatch], ignore_index=True)

with pd.ExcelWriter(OUTPUT_EXCEL, engine="openpyxl") as writer:
    equip_counts.to_excel(writer, sheet_name="설비별_파일수(b)", index=False)
    sheet2_df.to_excel(writer, sheet_name="파일명검증(c_d)", index=False)

print("\n===================== [e] RESULT =====================")
print(f"엑셀 파일 생성 완료 → {OUTPUT_EXCEL}")
print("sheet1: 설비별_파일수(b)")
print("sheet2: 파일명검증(c_d)")


===================== [e] RESULT =====================
엑셀 파일 생성 완료 → C:\Users\user\Desktop\a1_FCT_Vision_precheck_result.xlsx
sheet1: 설비별_파일수(b)
sheet2: 파일명검증(c_d)
